In [1]:
!pip install -q ultralytics

## Imports

In [8]:
from pathlib import Path
from collections import deque
from ultralytics import YOLO
import os
import cv2
import math
import numpy as np
from tracking_utils import (
    DEFAULT_FOCAL_LENGTH_PX,
    REAL_BALL_DIAMETER_CM,
    estimate_distance_cm,
    estimate_focal_length_px,
    get_best_model_path,
    get_data_yaml,
    get_dataset_dir,
    get_largest_ball,
)


In [9]:
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "tennis_dataset").exists() else Path.cwd().parent
DATASET_DIR = get_dataset_dir(PROJECT_ROOT)
DATA_YAML = get_data_yaml(PROJECT_ROOT)

print(DATA_YAML.exists())

True


## Step 1: Setup and Train

1. Run the install/import/path cells first.
2. Load `yolo11n.pt` as the starter model.
3. Train on `DATA_YAML`.
4. Run validation and check metrics before moving on.


In [4]:
model = YOLO("yolo11n.pt")

In [5]:
results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=8,
    device="mps",
    project="runs_notebook",
    name="tennis_ball_train"
)

Ultralytics 8.4.21 🚀 Python-3.10.12 torch-2.10.0 MPS (Apple M3 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/yuriy/Documents/Github/Study/od-dc-tennis/tennis_dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tennis_ball_train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overl

In [6]:
motrics = model.val(
    data=str(DATA_YAML),
    device="mps"
)

Ultralytics 8.4.21 🚀 Python-3.10.12 torch-2.10.0 MPS (Apple M3 Pro)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1901.8±767.1 MB/s, size: 4766.1 KB)
val: Scanning /Users/yuriy/Documents/Github/Study/od-dc-tennis/tennis_dataset/valid/labels.cache... 5 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5/5 873.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.6it/s 0.6s
                   all          5         24          1          1      0.995      0.885
Speed: 0.5ms preprocess, 8.4ms inference, 0.0ms loss, 4.0ms postprocess per image
Results saved to /Users/yuriy/Documents/Github/Study/od-dc-tennis/notebooks/runs/detect/val


In [9]:
TEST_IMAGE = "/Users/yuriy/Documents/Github/Study/od-dc-tennis/tennis_dataset/valid/images/02e40605-IMG_9016.jpeg"

preds = model.predict(
    source=TEST_IMAGE,
    device="mps",
    conf=0.25,
    save=True
)


image 1/1 /Users/yuriy/Documents/Github/Study/od-dc-tennis/tennis_dataset/valid/images/02e40605-IMG_9016.jpeg: 480x640 4 sports balls, 2 chairs, 293.6ms
Speed: 8.2ms preprocess, 293.6ms inference, 65.1ms postprocess per image at shape (1, 3, 480, 640)
Results saved to /Users/yuriy/Documents/Github/Study/od-dc-tennis/notebooks/runs/detect/predict5


In [10]:
best_model_path = get_best_model_path(PROJECT_ROOT)
print(best_model_path)
print(best_model_path.exists())


/Users/yuriy/Documents/Github/Study/od-dc-tennis/notebooks/runs/detect/runs_notebook/tennis_ball_train/weights/best.pt
True


In [11]:
best_model = YOLO(str(best_model_path))


In [13]:
results = best_model.predict(
    source=0,
    show=True,
    device="mps",
    conf=0.25,
    stream=True,
    verbose=False
)

for r in results:
    pass

1/1: 0... Success ✅ (inf frames of shape 1920x1080 at 15.00 FPS)



## Step 2: Quick Inference Check

1. Run prediction on one known image.
2. Confirm detections look correct.
3. Load `best.pt` and keep using it for all next steps.


In [1]:
import cv2
import numpy as np

best_model = YOLO(str(get_best_model_path(PROJECT_ROOT)))
FOCAL_LENGTH_PX = DEFAULT_FOCAL_LENGTH_PX  # replace this after calibration

cap = cv2.VideoCapture(0)

while True:
    ok, frame = cap.read()
    if not ok:
        break

    results = best_model(frame, verbose=False)
    annotated = results[0].plot()

    ball = get_largest_ball(results)

    if ball is not None:
        x1, y1, x2, y2, w, h = ball
        distance_cm = estimate_distance_cm(w, h, focal_length_px=FOCAL_LENGTH_PX, real_ball_diameter_cm=REAL_BALL_DIAMETER_CM)

        if distance_cm is not None:
            cv2.putText(
                annotated,
                f"{distance_cm:.1f} cm",
                (int(x1), int(y2) + 40),   # bottom of bounding box
                cv2.FONT_HERSHEY_SIMPLEX,
                1.5,                       # bigger text
                (0, 0, 255),               # red color (BGR)
                3                          # thicker text
            )

    cv2.imshow("Tennis Ball Distance", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 

In [2]:
results = best_model("calibration_image.jpg", verbose=False)
ball = get_largest_ball(results)

if ball is not None:
    _, _, _, _, w, h = ball
    observed_ball_diameter_px = (w + h) / 2.0

    focal_length_px = estimate_focal_length_px(
        observed_ball_diameter_px=observed_ball_diameter_px,
        known_distance_cm=50,
        real_ball_diameter_cm=REAL_BALL_DIAMETER_CM,
    )
    print("Estimated focal length:", focal_length_px)


SyntaxError: cannot assign to expression (2454248245.py, line 1)

## Step 3: Live Distance Estimation

1. Start webcam inference.
2. Detect the ball and estimate distance in cm.
3. Verify overlay values are stable while moving the ball.


## Step 4: Camera Calibration (Recommended)

1. Place the ball at a known distance (example: 50 cm).
2. Run calibration cell to estimate focal length.
3. Replace `FOCAL_LENGTH_PX` with the calibrated value.


In [1]:
import cv2
import time
import math
from collections import deque
from ultralytics import YOLO

best_model = YOLO(str(get_best_model_path(PROJECT_ROOT)))
FOCAL_LENGTH_PX = DEFAULT_FOCAL_LENGTH_PX

# How many recent tracked points to keep
TRACK_HISTORY = 10

# How far ahead to predict
PREDICT_STEPS = 12
PREDICT_DT = 0.08  # seconds between predicted points

# Friction factor for rolling ball prediction
FRICTION = 0.93

trajectory = deque(maxlen=TRACK_HISTORY)


def draw_text_with_outline(img, text, pos, font_scale=1.0, color=(255, 255, 255), thickness=2):
    cv2.putText(
        img,
        text,
        pos,
        cv2.FONT_HERSHEY_SIMPLEX,
        font_scale,
        (0, 0, 0),
        thickness + 3,
        cv2.LINE_AA
    )
    cv2.putText(
        img,
        text,
        pos,
        cv2.FONT_HERSHEY_SIMPLEX,
        font_scale,
        color,
        thickness,
        cv2.LINE_AA
    )


def draw_info_panel(frame, distance=None, speed=None, vz=None):
    overlay = frame.copy()

    panel_x, panel_y = 15, 15
    panel_width, panel_height = 360, 150

    # Background panel
    cv2.rectangle(
        overlay,
        (panel_x, panel_y),
        (panel_x + panel_width, panel_y + panel_height),
        (20, 20, 20),
        -1
    )

    # Blend transparent panel
    alpha = 0.55
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)

    # Optional border
    cv2.rectangle(
        frame,
        (panel_x, panel_y),
        (panel_x + panel_width, panel_y + panel_height),
        (80, 80, 80),
        2
    )

    # Title
    draw_text_with_outline(
        frame,
        "Tennis Ball Tracker",
        (panel_x + 15, panel_y + 32),
        font_scale=0.9,
        color=(0, 0, 255),
        thickness=2
    )

    # Separator line
    cv2.line(
        frame,
        (panel_x + 12, panel_y + 45),
        (panel_x + panel_width - 12, panel_y + 45),
        (90, 90, 90),
        1
    )

    distance_text = f"Distance : {distance:.1f} cm" if distance is not None else "Distance : --"
    speed_text = f"Speed    : {speed:.1f} px/s" if speed is not None else "Speed    : --"
    vz_text = f"Z speed  : {vz:.1f} cm/s" if vz is not None else "Z speed  : --"

    lines = [distance_text, speed_text, vz_text]
    y = panel_y + 75

    for line in lines:
        draw_text_with_outline(
            frame,
            line,
            (panel_x + 15, y),
            font_scale=0.8,
            color=(255, 255, 255),
            thickness=2
        )
        y += 30


def estimate_velocity(track):
    """
    Estimate velocity from the last two trajectory points.
    Each point: (timestamp, cx, cy, distance_cm)
    Returns vx, vy in pixels/sec and vz in cm/sec
    """
    if len(track) < 2:
        return None

    t1, x1, y1, z1 = track[-2]
    t2, x2, y2, z2 = track[-1]

    dt = t2 - t1
    if dt <= 1e-6:
        return None

    vx = (x2 - x1) / dt
    vy = (y2 - y1) / dt
    vz = (z2 - z1) / dt

    return vx, vy, vz


def draw_predicted_trajectory(img, cx, cy, vx, vy):
    """
    Draw future rolling path as red dots.
    Friction gradually reduces velocity.
    """
    px = float(cx)
    py = float(cy)
    vxp = float(vx)
    vyp = float(vy)

    for _ in range(PREDICT_STEPS):
        px += vxp * PREDICT_DT
        py += vyp * PREDICT_DT

        cv2.circle(img, (int(px), int(py)), 6, (0, 0, 255), -1)

        vxp *= FRICTION
        vyp *= FRICTION


cap = cv2.VideoCapture(0)

while True:
    ok, frame = cap.read()
    if not ok:
        break

    current_time = time.time()

    results = best_model(frame, verbose=False)
    annotated = results[0].plot()

    ball = get_largest_ball(results)

    distance_cm = None
    speed_px = None
    vz = None

    if ball is not None:
        x1, y1, x2, y2, w, h = ball

        # Ball center in pixels
        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0

        # Distance estimation from apparent size
        ball_diameter_px = (w + h) / 2.0
        if ball_diameter_px > 0:
            distance_cm = estimate_distance_cm(w, h, focal_length_px=FOCAL_LENGTH_PX, real_ball_diameter_cm=REAL_BALL_DIAMETER_CM)

            if distance_cm is None:
                continue

            # Save tracking point
            trajectory.append((current_time, cx, cy, distance_cm))

            # Draw current center
            cv2.circle(annotated, (int(cx), int(cy)), 6, (255, 0, 0), -1)

            # Draw track history
            for i in range(1, len(trajectory)):
                _, x_prev, y_prev, _ = trajectory[i - 1]
                _, x_curr, y_curr, _ = trajectory[i]
                cv2.line(
                    annotated,
                    (int(x_prev), int(y_prev)),
                    (int(x_curr), int(y_curr)),
                    (255, 255, 0),
                    2
                )

            # Estimate current velocity
            velocity = estimate_velocity(trajectory)

            if velocity is not None:
                vx, vy, vz = velocity
                speed_px = math.sqrt(vx * vx + vy * vy)

                # Draw predicted rolling trajectory
                draw_predicted_trajectory(annotated, cx, cy, vx, vy)

    else:
        trajectory.clear()

    # Draw clean top-left info panel every frame
    draw_info_panel(
        annotated,
        distance=distance_cm,
        speed=speed_px,
        vz=vz
    )

    cv2.imshow("Tennis Ball Rolling Prediction", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 

## Step 5: Tracking and Validation

1. Run live rolling tracker to inspect trajectory behavior.
2. Run recorded-video tracker for repeatable validation.
3. Review output video and tune thresholds if needed.


In [12]:
INPUT_VIDEO = str(DATASET_DIR.joinpath("valid/videos/4.mp4"))
OUTPUT_VIDEO = str(DATASET_DIR.joinpath("valid/videos/4_tracked_output.mp4"))
MODEL_PATH = str(get_best_model_path(PROJECT_ROOT))
best_model = YOLO(MODEL_PATH)
FOCAL_LENGTH_PX = DEFAULT_FOCAL_LENGTH_PX
TRACK_HISTORY = 20
MAX_TRACK_MISSES = 12
MATCH_DISTANCE_THRESHOLD = 120.0

PREDICT_DT = 0.08
PREDICT_MAX_STEPS = 50
FRICTION = 0.93
MIN_STOP_SPEED_PX = 8.0

# Stable stop-point logic
STOP_UPDATE_INTERVAL = 1.0       # seconds
STOP_SPEED_CHANGE_THRESH = 40.0  # px/s
MIN_FREEZE_STOP_SPEED_PX = 20.0  # below this, don't keep updating stop point

# Velocity smoothing
VELOCITY_WINDOW = 5

def draw_text(img, text, pos, scale=0.8, color=(255, 255, 255), thick=2):
    cv2.putText(
        img, text, pos, cv2.FONT_HERSHEY_SIMPLEX,
        scale, (0, 0, 0), thick + 3, cv2.LINE_AA
    )
    cv2.putText(
        img, text, pos, cv2.FONT_HERSHEY_SIMPLEX,
        scale, color, thick, cv2.LINE_AA
    )


def draw_info_panel(frame, primary_track=None, track_count=0):
    overlay = frame.copy()

    x, y = 15, 15
    w, h = 430, 195

    cv2.rectangle(overlay, (x, y), (x + w, y + h), (20, 20, 20), -1)
    alpha = 0.55
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)

    cv2.rectangle(frame, (x, y), (x + w, y + h), (80, 80, 80), 2)

    draw_text(frame, "Tennis Ball Multi-Tracker", (x + 15, y + 32), 0.9, (0, 0, 255), 2)
    cv2.line(frame, (x + 10, y + 45), (x + w - 10, y + 45), (90, 90, 90), 1)

    if primary_track is None:
        lines = [
            f"Active tracks : {track_count}",
            "Primary ID    : --",
            "Distance      : --",
            "Speed         : --",
            "Z speed       : --",
            "Stop dist     : --",
        ]
    else:
        dist = f"{primary_track.distance_cm:.1f} cm" if primary_track.distance_cm is not None else "--"
        spd = f"{primary_track.speed_cm_s:.1f} cm/s" if primary_track.speed_cm_s is not None else "--"
        vz = f"{primary_track.vz_cm_s:.1f} cm/s" if primary_track.vz_cm_s is not None else "--"
        stop_dist = f"{primary_track.stop_distance_cm:.1f} cm" if primary_track.stop_distance_cm is not None else "--"

        lines = [
            f"Active tracks : {track_count}",
            f"Primary ID    : {primary_track.track_id}",
            f"Distance      : {dist}",
            f"Speed         : {spd}",
            f"Z speed       : {vz}",
            f"Stop dist     : {stop_dist}",
        ]

    yy = y + 72
    for line in lines:
        draw_text(frame, line, (x + 15, yy), 0.75, (255, 255, 255), 2)
        yy += 26

def get_ball_detections(results):
    boxes = results[0].boxes
    detections = []

    if boxes is None or len(boxes) == 0:
        return detections

    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0].cpu().numpy()) if box.conf is not None else 1.0

        w = x2 - x1
        h = y2 - y1
        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0
        diameter_px = (w + h) / 2.0

        if diameter_px <= 0:
            continue

        distance_cm = (REAL_BALL_DIAMETER_CM * FOCAL_LENGTH_PX) / diameter_px

        detections.append({
            "bbox": (x1, y1, x2, y2),
            "center": (cx, cy),
            "w": w,
            "h": h,
            "diameter_px": diameter_px,
            "distance_cm": distance_cm,
            "conf": conf,
        })

    return detections

def create_kalman_filter(x, y):
    kf = cv2.KalmanFilter(4, 2)
    kf.transitionMatrix = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ], dtype=np.float32)

    kf.measurementMatrix = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0]
    ], dtype=np.float32)

    kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
    kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 0.5
    kf.errorCovPost = np.eye(4, dtype=np.float32)

    kf.statePost = np.array([[x], [y], [0], [0]], dtype=np.float32)
    return kf


def estimate_velocity_from_history(history, window=VELOCITY_WINDOW):
    if len(history) < 2:
        return None

    points = list(history)[-window:]
    if len(points) < 2:
        return None

    velocities_x = []
    velocities_y = []
    velocities_z = []

    for i in range(1, len(points)):
        t1, x1, y1, z1 = points[i - 1]
        t2, x2, y2, z2 = points[i]
        dt = t2 - t1

        if dt <= 1e-6:
            continue

        velocities_x.append((x2 - x1) / dt)
        velocities_y.append((y2 - y1) / dt)
        velocities_z.append((z2 - z1) / dt)

    if not velocities_x:
        return None

    vx = float(np.mean(velocities_x))
    vy = float(np.mean(velocities_y))
    vz = float(np.mean(velocities_z))

    return vx, vy, vz


def predict_path_and_stop(cx, cy, vx, vy, distance_cm):
    path = []
    px = float(cx)
    py = float(cy)
    vxp = float(vx)
    vyp = float(vy)

    total_px_distance = 0.0

    for _ in range(PREDICT_MAX_STEPS):
        step_dx = vxp * PREDICT_DT
        step_dy = vyp * PREDICT_DT

        px += step_dx
        py += step_dy
        total_px_distance += math.sqrt(step_dx ** 2 + step_dy ** 2)

        path.append((int(px), int(py)))

        vxp *= FRICTION
        vyp *= FRICTION

        if math.sqrt(vxp ** 2 + vyp ** 2) < MIN_STOP_SPEED_PX:
            break

    stop_point = path[-1] if path else None

    stop_distance_cm = None
    if distance_cm is not None:
        stop_distance_cm = (total_px_distance * distance_cm) / FOCAL_LENGTH_PX

    return path, stop_point, stop_distance_cm


class Track:
    def __init__(self, track_id, detection, timestamp):
        cx, cy = detection["center"]
        self.track_id = track_id
        self.kf = create_kalman_filter(cx, cy)

        self.bbox = detection["bbox"]
        self.distance_cm = detection["distance_cm"]
        self.diameter_px = detection["diameter_px"]
        self.conf = detection["conf"]

        self.history = deque(maxlen=TRACK_HISTORY)
        self.history.append((timestamp, cx, cy, self.distance_cm))

        self.missed = 0
        self.last_time = timestamp

        self.vx_px_s = 0.0
        self.vy_px_s = 0.0
        self.vz_cm_s = 0.0
        self.speed_px_s = 0.0
        self.speed_cm_s = 0.0

        self.predicted_path = []
        self.stop_point = None
        self.stop_distance_cm = None

        # Stable stop-point state
        self.last_stop_update_time = timestamp
        self.last_stop_speed_px_s = 0.0

    def predict(self):
        pred = self.kf.predict()
        px = float(pred[0][0])
        py = float(pred[1][0])
        return px, py

    def update(self, detection, timestamp):
        cx, cy = detection["center"]
        measurement = np.array([[np.float32(cx)], [np.float32(cy)]])
        self.kf.correct(measurement)

        self.bbox = detection["bbox"]
        self.distance_cm = detection["distance_cm"]
        self.diameter_px = detection["diameter_px"]
        self.conf = detection["conf"]

        self.history.append((timestamp, cx, cy, self.distance_cm))
        self.missed = 0
        self.last_time = timestamp

        self._compute_velocity_and_prediction()

    def mark_missed(self):
        self.missed += 1

    def current_smoothed_center(self):
        state = self.kf.statePost
        return float(state[0][0]), float(state[1][0])

    def _compute_velocity_and_prediction(self):
        velocity = estimate_velocity_from_history(self.history, window=VELOCITY_WINDOW)
        if velocity is None:
            self.vx_px_s = 0.0
            self.vy_px_s = 0.0
            self.vz_cm_s = 0.0
            self.speed_px_s = 0.0
            self.speed_cm_s = 0.0
            self.predicted_path = []
            return

        hist_vx, hist_vy, hist_vz = velocity

        # Blend history-based velocity with Kalman velocity
        state = self.kf.statePost
        kf_vx = float(state[2][0])
        kf_vy = float(state[3][0])

        self.vx_px_s = 0.7 * hist_vx + 0.3 * kf_vx
        self.vy_px_s = 0.7 * hist_vy + 0.3 * kf_vy
        self.vz_cm_s = hist_vz

        self.speed_px_s = math.sqrt(self.vx_px_s ** 2 + self.vy_px_s ** 2)
        self.speed_cm_s = (self.speed_px_s * self.distance_cm) / FOCAL_LENGTH_PX if self.distance_cm is not None else 0.0

        # Update live predicted path every frame
        _, x2, y2, _ = self.history[-1]
        self.predicted_path, _, _ = predict_path_and_stop(
            x2, y2, self.vx_px_s, self.vy_px_s, self.distance_cm
        )

        # Stable stop point: only update every interval or after strong speed change
        current_time = self.history[-1][0]
        speed_change = abs(self.speed_px_s - self.last_stop_speed_px_s)

        should_update_stop = (
            self.stop_point is None
            or (
                self.speed_px_s >= MIN_FREEZE_STOP_SPEED_PX and (
                    (current_time - self.last_stop_update_time) >= STOP_UPDATE_INTERVAL
                    or speed_change >= STOP_SPEED_CHANGE_THRESH
                )
            )
        )

        if should_update_stop:
            _, self.stop_point, self.stop_distance_cm = predict_path_and_stop(
                x2, y2, self.vx_px_s, self.vy_px_s, self.distance_cm
            )
            self.last_stop_update_time = current_time
            self.last_stop_speed_px_s = self.speed_px_s

tracks = {}
next_track_id = 1


def associate_detections_to_tracks(detections, tracks_dict):
    unmatched_detections = set(range(len(detections)))
    unmatched_tracks = set(tracks_dict.keys())
    matches = []

    if not detections or not tracks_dict:
        return matches, unmatched_detections, unmatched_tracks

    candidates = []

    for tid, track in tracks_dict.items():
        pred_x, pred_y = track.predict()
        for det_idx, det in enumerate(detections):
            cx, cy = det["center"]
            dist = math.sqrt((cx - pred_x) ** 2 + (cy - pred_y) ** 2)
            candidates.append((dist, tid, det_idx))

    candidates.sort(key=lambda x: x[0])

    used_tracks = set()
    used_detections = set()

    for dist, tid, det_idx in candidates:
        if dist > MATCH_DISTANCE_THRESHOLD:
            continue
        if tid in used_tracks or det_idx in used_detections:
            continue

        matches.append((tid, det_idx))
        used_tracks.add(tid)
        used_detections.add(det_idx)

    unmatched_tracks -= used_tracks
    unmatched_detections -= used_detections

    return matches, unmatched_detections, unmatched_tracks


def choose_primary_track(tracks_dict):
    visible_tracks = [t for t in tracks_dict.values() if t.missed == 0 and t.distance_cm is not None]
    if not visible_tracks:
        return None
    return min(visible_tracks, key=lambda t: t.distance_cm)

cap = cv2.VideoCapture(INPUT_VIDEO)

if not cap.isOpened():
    print("Failed to open video")
    raise SystemExit

fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 30

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("Video:", width, "x", height, "FPS:", fps)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

while True:
    ok, frame = cap.read()
    if not ok:
        break

    current_time = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0

    results = best_model(frame, verbose=False)
    annotated = results[0].plot()

    detections = get_ball_detections(results)

    matches, unmatched_dets, unmatched_tracks = associate_detections_to_tracks(detections, tracks)

    for tid, det_idx in matches:
        tracks[tid].update(detections[det_idx], current_time)

    for tid in unmatched_tracks:
        tracks[tid].mark_missed()

    for det_idx in unmatched_dets:
        det = detections[det_idx]
        tracks[next_track_id] = Track(next_track_id, det, current_time)
        next_track_id += 1

    dead_ids = [tid for tid, tr in tracks.items() if tr.missed > MAX_TRACK_MISSES]
    for tid in dead_ids:
        del tracks[tid]

    # Draw all visible tracks
    for tid, track in tracks.items():
        if track.missed > 0:
            continue

        x1, y1, x2, y2 = track.bbox
        cx, cy = track.current_smoothed_center()

        # Smoothed center
        cv2.circle(annotated, (int(cx), int(cy)), 6, (255, 0, 0), -1)

        # Track ID
        draw_text(
            annotated,
            f"ID {track.track_id}",
            (int(x1), max(25, int(y1) - 10)),
            0.7,
            (0, 255, 255),
            2
        )

        # History
        if len(track.history) >= 2:
            for i in range(1, len(track.history)):
                _, xp, yp, _ = track.history[i - 1]
                _, xc, yc, _ = track.history[i]
                cv2.line(
                    annotated,
                    (int(xp), int(yp)),
                    (int(xc), int(yc)),
                    (255, 255, 0),
                    2
                )

        # Predicted path
        if track.predicted_path:
            for i in range(1, len(track.predicted_path)):
                cv2.line(
                    annotated,
                    track.predicted_path[i - 1],
                    track.predicted_path[i],
                    (0, 0, 255),
                    2
                )
            for pt in track.predicted_path[::3]:
                cv2.circle(annotated, pt, 4, (0, 0, 255), -1)

        # Stable stop point
        if track.stop_point is not None:
            cv2.circle(annotated, track.stop_point, 9, (0, 255, 0), 2)
            draw_text(
                annotated,
                "STOP",
                (track.stop_point[0] + 8, track.stop_point[1] - 8),
                0.6,
                (0, 255, 0),
                2
            )

    primary_track = choose_primary_track(tracks)
    draw_info_panel(annotated, primary_track=primary_track, track_count=len(tracks))

    out.write(annotated)
    cv2.imshow("Ball Tracking", annotated)

    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
        break

cap.release()
out.release()
cv2.destroyAllWindows()
cv2.waitKey(1)


Video: 3840 x 2160 FPS: 59.94005994005994


-1